### Middleware

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()
# os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

### Summarization Middleware

In [18]:
import os
from dotenv import load_dotenv
from langchain.messages import HumanMessage, SystemMessage
# from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import MemorySaver

load_dotenv()

# Instantiate explicit Chat models (use valid model names like gemini-1.5-flash or gemini-2.0-flash)
# main_model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")
# summary_model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")
# llm = ChatGoogleGenerativeAI(
#     model="gemini-3.8-flash",
#     google_api_key=os.getenv("GOOGLE_API_KEY")
# )
os.environ["OPENROUTER_API_KEY"]=os.getenv("OPENROUTER_API_KEY")
llm=init_chat_model("nvidia/nemotron-3-ultra-550b-a55b:free",model_provider="openrouter")

agent = create_agent(
    model=llm,
    checkpointer=MemorySaver(),
    # memory=MemorySaver(), #gives error if used
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("messages", 10),
            keep=("messages", 4)
        )
    ],
)

In [19]:
## Run with thread id
config = {"configurable":{"thread_id":"test-user-1"}}

In [15]:
# Test data
questions = [
    "what is 2+2?",
    "what is 3-3?",
    "what is 4*4?",
    "what is 5/5?",
    "what is 6/0?"
]

for question in questions:
    messages = [SystemMessage(content="You are a helpful assistant."), HumanMessage(content=question)]
    response = agent.invoke({"messages": messages}, config=config)
    print(f"Question: {question}")
    print(f"Message: {response}")
    print(f"Message: {len(response['messages'])}")


Question: what is 2+2?
Message: {'messages': [SystemMessage(content='You are a helpful assistant.', additional_kwargs={}, response_metadata={}, id='65384da1-21e3-45bc-97e5-9d8a4229051b'), HumanMessage(content='what is 2+2?', additional_kwargs={}, response_metadata={}, id='018d3c1b-5c57-4ecd-95ab-1f4591f66ed9'), AIMessage(content='2 + 2 = 4', additional_kwargs={'reasoning_content': 'The user is asking for the sum of 2 and 2.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The user is asking for the sum of 2 and 2.'}]}, response_metadata={'model_name': 'nvidia/nemotron-3-ultra-550b-a55b:free', 'id': 'gen-1790916672-3fzB1dYBD01TJXojGS4z', 'created': 1790916672, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0, 'cost_details': {'upstream_inference_completions_cost': 0.0, 'upstream_inference_prompt_cost': 0.0, 'upstream_inference_cost': 0.0}}, id='lc_run--01a0faf3-b231-7652-874b-4

### Token Size

In [21]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns response to use less tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent=create_agent(
    model=llm,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("tokens",550),
            keep=("tokens",200),
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}} #Identifies the conversation thread. Useful if you want multiple independent sessions.

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token

In [22]:
## Run test
cities = ["New York", "Los Angeles", "Chicago", "Houston", "Phoenix"]
for city in cities:
    response = agent.invoke({"messages": [HumanMessage(content=f"Search hotels in {city}")]}, config=config)
    tokens = count_tokens(response['messages'])

    print(f"City: {city}")
    print(f"Message: {response}")
    print(f"(response['messages']")
    print(f"{city}:~{tokens} tokens,{len(response['messages'])} messages")
    print(f"Tokens: {count_tokens(response['messages'])}")


City: New York
Message: {'messages': [HumanMessage(content='Search hotels in New York', additional_kwargs={}, response_metadata={}, id='8883255c-435f-464b-9c00-29bffb518300'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to search for hotels in New York. I need to call the search_hotels function with city parameter "New York". Let me do that.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The user wants to search for hotels in New York. I need to call the search_hotels function with city parameter "New York". Let me do that.'}]}, response_metadata={'model_name': 'nvidia/nemotron-3-ultra-550b-a55b:free', 'id': 'gen-1790920367-cSOj6YCbNofiuNaGtV1r', 'created': 1790920367, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0, 'cost_details': {'upstream_inference_completions_cost': 0.0, 'upstream_inference_prompt_cost': 0.0, 'upstream_inference

In [23]:
response['messages']

[HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\nThe user wants to search for hotels in Houston and refine the search with specific criteria.\n\n## SUMMARY\n\nA hotel search was performed for Houston with no filters applied (no dates, budget, neighborhood, guests, or amenities specified). Three hotels were returned: Grand Hotel (5-star, $350/night, spa/pool/gym), City Inn (4-star, $180/night, business center), Budget Stay (3-star, $75/night, free WiFi). These results were presented to the user. The user previously searched for Chicago but switched to Houston. Clarifying questions have been asked to refine the Houston search.\n\n## ARTIFACTS\n\nNone - no files or artifacts created. Hotel search results are ephemeral tool output.\n\n## NEXT STEPS\n\nWait for user to respond to clarifying questions to refine the Houston hotel search:\n- Check-in and check-out dates\n- Number of guests/rooms\n- Preferred neighborhood/area in Houston\n- Budget ra

### Fractions

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
    model="llm",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="llm",
            trigger=("fraction", 0.005),  # 0.5% of 160k tokens(llm context) = ~640 tokens, based on context of llm model
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  # gpt-4o-mini context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

### Human in the Loop Middleware

In [25]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read email by its ID."""
    return f"Email content for ID: {email_id}."


def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject {subject}"

In [45]:
agent = create_agent(
    model=llm,
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","reject", "edit"]
                },
                "read_email_tool":False
            }
        )
    ]
)


In [46]:
config = {"configurable": {"thread_id": "test-approve"}}
# step 1: request
result = agent.invoke(
    {"messages": [HumanMessage(content="Please send an email to sanaullah@email.com with subject 'Meeting Reminder' and body 'Don't forget our meeting tomorrow at 10 AM.'")]},
    config=config
)
print(result)

{'messages': [HumanMessage(content="Please send an email to sanaullah@email.com with subject 'Meeting Reminder' and body 'Don't forget our meeting tomorrow at 10 AM.'", additional_kwargs={}, response_metadata={}, id='f459f658-b90a-4122-8b96-8eb6f2875e9d'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants me to send an email to sanaullah@email.com with subject "Meeting Reminder" and body "Don\'t forget our meeting tomorrow at 10 AM." I have the send_email_tool available. I\'ll use that.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The user wants me to send an email to sanaullah@email.com with subject "Meeting Reminder" and body "Don\'t forget our meeting tomorrow at 10 AM." I have the send_email_tool available. I\'ll use that.'}]}, response_metadata={'model_name': 'nvidia/nemotron-3-ultra-550b-a55b:free', 'id': 'gen-1790924398-76W1tdWp2NBgIAgG686p', 'created': 1790924398, 'object': 'chat.completion', 'finish_re

In [47]:
from langgraph.types import Command
# Step 2: Human review (simulated)
if "__interrupt__" in result:
    print("Paused! Approvi g...")
    # Human review and approval
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    print(result)

Paused! Approvi g...
{'messages': [HumanMessage(content="Please send an email to sanaullah@email.com with subject 'Meeting Reminder' and body 'Don't forget our meeting tomorrow at 10 AM.'", additional_kwargs={}, response_metadata={}, id='f459f658-b90a-4122-8b96-8eb6f2875e9d'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants me to send an email to sanaullah@email.com with subject "Meeting Reminder" and body "Don\'t forget our meeting tomorrow at 10 AM." I have the send_email_tool available. I\'ll use that.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The user wants me to send an email to sanaullah@email.com with subject "Meeting Reminder" and body "Don\'t forget our meeting tomorrow at 10 AM." I have the send_email_tool available. I\'ll use that.'}]}, response_metadata={'model_name': 'nvidia/nemotron-3-ultra-550b-a55b:free', 'id': 'gen-1790924398-76W1tdWp2NBgIAgG686p', 'created': 1790924398, 'object': 'chat.co

In [48]:
result

{'messages': [HumanMessage(content="Please send an email to sanaullah@email.com with subject 'Meeting Reminder' and body 'Don't forget our meeting tomorrow at 10 AM.'", additional_kwargs={}, response_metadata={}, id='f459f658-b90a-4122-8b96-8eb6f2875e9d'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants me to send an email to sanaullah@email.com with subject "Meeting Reminder" and body "Don\'t forget our meeting tomorrow at 10 AM." I have the send_email_tool available. I\'ll use that.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The user wants me to send an email to sanaullah@email.com with subject "Meeting Reminder" and body "Don\'t forget our meeting tomorrow at 10 AM." I have the send_email_tool available. I\'ll use that.'}]}, response_metadata={'model_name': 'nvidia/nemotron-3-ultra-550b-a55b:free', 'id': 'gen-1790924398-76W1tdWp2NBgIAgG686p', 'created': 1790924398, 'object': 'chat.completion', 'finish_

### Reject

In [58]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read email by its ID."""
    return f"Email content for ID: {email_id}."


def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject {subject}"

agent = create_agent(
    model=llm,
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","reject", "edit"]
                },
                "read_email_tool":False
            }
        )
    ]
)


In [59]:
config = {"configurable": {"thread_id": "test-reject"}}  # Thread id is changed here
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config)
print(result)

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='07b1ceca-5cfe-486a-8237-e32332ee16e2'), AIMessage(content='', additional_kwargs={'reasoning_content': "The user wants to send an email to john@test.com with subject 'Hello' and body 'How are you?'. I need to use the send_email_tool function. Let me check the required parameters: recipient, subject, body. All are provided. I'll call the function.", 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': "The user wants to send an email to john@test.com with subject 'Hello' and body 'How are you?'. I need to use the send_email_tool function. Let me check the required parameters: recipient, subject, body. All are provided. I'll call the function."}]}, response_metadata={'model_name': 'nvidia/nemotron-3-ultra-550b-a55b:free', 'id': 'gen-1790925853-6BzxAbugskV1DNgZHjB8', 'created': 1790925853, 'object

In [60]:
# Step 2: Reject
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: I attempted to send the email to john@test.com with subject "Hello" and body "How are you?", but the action was rejected. 

Would you like me to try again, or is there something else I can help you with regarding email?


In [61]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='07b1ceca-5cfe-486a-8237-e32332ee16e2'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "The user wants to send an email to john@test.com with subject 'Hello' and body 'How are you?'. I need to use the send_email_tool function. Let me check the required parameters: recipient, subject, body. All are provided. I'll call the function.", 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': "The user wants to send an email to john@test.com with subject 'Hello' and body 'How are you?'. I need to use the send_email_tool function. Let me check the required parameters: recipient, subject, body. All are provided. I'll call the function."}]}, response_metadata={'model_name': 'nvidia/nemotron-3-ultra-550b-a55b:free', 'id': 'gen-1790925853-6BzxAbugskV1DNgZHjB8', 'created': 1790925853, 'obje

### Edit

In [79]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read email by its ID."""
    return f"Email content for ID: {email_id}."


def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject {subject}"

agent = create_agent(
    model=llm,
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","reject", "edit"]
                },
                "read_email_tool":False
            }
        )
    ]
)


In [80]:
config = {"configurable": {"thread_id": "test-edit"}}  # Thread id is changed here
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@test.com with subject 'Test' and body 'How are you?'")]},
    config=config)
print(result)

{'messages': [HumanMessage(content="Send email to wrong@test.com with subject 'Test' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='1c05bcbc-8141-43dc-900e-54f844e7d933'), AIMessage(content='', additional_kwargs={'reasoning_content': "The user wants to send an email to wrong@test.com with subject 'Test' and body 'How are you?'. I have the send_email_tool available. I need to call it with the recipient, subject, and body parameters. Let me do that.", 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': "The user wants to send an email to wrong@test.com with subject 'Test' and body 'How are you?'. I have the send_email_tool available. I need to call it with the recipient, subject, and body parameters. Let me do that."}]}, response_metadata={'model_name': 'nvidia/nemotron-3-ultra-550b-a55b:free', 'id': 'gen-1790926911-7EFxREdfcV5o4jbNl2FQ', 'created': 1790926911, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'lo

In [84]:
# Step 2: Edit and approve
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")
    
   # Resume the graph with an edited action payload
    agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool", # The tool name to run
                            "args": {                  # The modified parameters
                                "to": "alice@test.com", 
                                "subject": "Updated Subject Line", 
                                "body": "This text was modified by a human reviewer."
                            }
                        }
                    }
                ]
            }
        ),
        config=config # Must match the original thread ID to resume the state
    )
    print(f"✏️ Result: {result['messages'][-1].content}")

⏸️ Paused! Editing...


KeyError: 'edited_action'